# Search Hardening

Notebook 12's `POST /search/` is pure CLIP: encode the query with the text encoder, run HNSW over pgvector, return top-k. This is the correct design for "photos of dogs on a beach" — a query whose semantics live in image content. It is the wrong design for two other classes of query that show up the moment a real user types:

- **Structured queries that CLIP cannot rank.** "Bondi, July 2023" — CLIP has no concept of Bondi and no notion of month. The query parses to a structured filter (`location=Bondi`, `date=[2023-07, 2023-08)`) and a freeform residual. The structured filter candidates should all be retrieved; CLIP ranks only inside that subset.
- **"Burst shooting" queries that CLIP over-ranks.** "Ben on the beach" returns Ben, but if your photographer fired 8 fps for ten seconds you get ten near-identical frames in the top-k and zero diversity. Top-k is wasted.

Notebook 13's `SmartSearchHandler` parsed the natural-language query into structured fields and then *discarded* the structured-field contribution to ranking — it used the parsed filter only as a SQL `WHERE`, then handed the candidates to CLIP. [The structured information was a prefilter, not a scorer]{.mark}. This notebook closes that gap and two others: we make the search endpoint hybrid (CLIP similarity + structured match score), add face-presence re-ranking for queries that mention a person, cache the CLIP text embedding per query in Redis so repeated queries skip the encoder, and apply MMR-style dedup to the top-k so burst frames are not over-represented.

<br>

**The flow.**

```
query text
   ├─ (a) parse with GPT-4o-mini  → SearchQuery {person, location, start, end, freeform}
   ├─ (b) embed freeform (or full text) with CLIP  → query_vec  (cached in Redis)
   ├─ (c) SQL: SELECT photo_id WHERE person/loc/date filter matches  → candidates
   ├─ (d) pgvector: rank candidates by query_vec cosine  → ranked_candidates
   ├─ (e) face-presence re-rank: boost photos where a detected face matches `person`
   ├─ (f) hybrid score combine: 0.7 * CLIP_sim + 0.3 * structured_match + face_boost
   ├─ (g) MMR dedup over top-k+oversample
   └─ (h) presign URLs (cached, see PHT:03), build response
```

Stages (b) and (h) are individually idempotent and cacheable. Stages (a,c,d) come from notebook 12 and 13 unchanged; stages (e,f,g) are the new work of this notebook. Cache hits at (b) turn the CLIP encode step — ~20 ms on CPU per `12-photo-app.ipynb:[3]` — into a sub-millisecond Redis read for repeat queries, which matters more than it sounds: users re-run the same query, search autocomplete fires partial queries, and saved searches re-execute every view.

---

## The SearchQuery Model and GPT Parse

We reuse notebook 13's `SearchQuery` shape — `person_name`, `location_hint`, `start_date`, `end_date`, `freeform` — and extend it with `confidence` (the model's self-assessed parse quality, used to decide whether to skip stage (c) entirely when the parse is weak).


In [ ]:
import json
from datetime import date
from pydantic import BaseModel
from unittest.mock import AsyncMock, MagicMock


class SearchQuery(BaseModel):
    """Structured fields parsed from a natural-language search query.
    Reused from notebook 13 and extended with `confidence` for parse quality.
    """
    person_name:    str | None = None
    location_hint:  str | None = None
    start_date:     date | None = None
    end_date:       date | None = None
    freeform:       str | None = None
    confidence:     float = 0.0   # model self-assessed parse quality [0,1]


class ParsedQuery(BaseModel):
    """What GPT-4o-mini returns from the parse step. Mirrors SearchQuery plus
    a `parse_notes` field the UI can show when the parse is ambiguous."""
    query:        SearchQuery
    parse_notes:  str = ""


async def parse_query(llm_client, text: str) -> ParsedQuery:
    """Parse natural language into a structured SearchQuery.

    Uses GPT-4o-mini with `response_format={"type": "json_object"}` for
    constrained decoding. The structured output is validated against
    SearchQuery; malformed output raises a Pydantic ValidationError.
    """
    sys_msg = (
        "Extract a search filter from the user's photo query. Return JSON with "
        "keys: person_name (string|null), location_hint (string|null), "
        "start_date (YYYY-MM-DD|null), end_date (YYYY-MM-DD|null), "
        "freeform (any residual text the structured fields did not capture), "
        "confidence (your self-assessed parse quality, 0.0 to 1.0). "
        "Examples: 'Ben at Bondi in July 2023' → "
        '{"person_name":"Ben","location_hint":"Bondi","start_date":"2023-07-01",'
        ' "end_date":"2023-08-01","freeform":null,"confidence":0.9}.'
    )
    resp = await llm_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": sys_msg},
            {"role": "user",   "content": text},
        ],
        response_format={"type": "json_object"},
    )
    raw = resp.choices[0].message.content
    parsed = json.loads(raw)
    return ParsedQuery(
        query=SearchQuery.model_validate(parsed),
        parse_notes=parsed.get("notes", ""),
    )


# Demonstrate against a mock LLM.
def _mock_llm(response_text: str):
    choice = MagicMock(); choice.message.content = response_text
    completion = MagicMock(); completion.choices = [choice]
    client = MagicMock()
    client.chat.completions.create = AsyncMock(return_value=completion)
    return client


mock_llm = _mock_llm(json.dumps({
    "person_name": "Ben", "location_hint": "Bondi",
    "start_date": "2023-07-01", "end_date": "2023-08-01",
    "freeform": None, "confidence": 0.9,
}))
pq = await parse_query(mock_llm, "Ben at Bondi in July 2023")
print(pq.model_dump_json(indent=2))


## Query → Embedding Cache

Stage (b): embed the residual `freeform` (or the full query text when no structured parse is possible) with CLIP. CLIP text encoding is ~20 ms on CPU; caching the embedding per query string drops this to a sub-millisecond Redis hit on repeats. The cache key is `sha256(query_text)`; the value is the float32 embedding serialized as raw bytes (numpy `.tobytes()`), kept under a **24 h TTL**. Users tend to re-run the same query within a day; saved searches re-execute on every view.


In [ ]:
import hashlib
import time
import numpy as np

try:
    import redis.asyncio as aioredis
    _HAS_REDIS = True
except ImportError:
    _HAS_REDIS = False


_EMBED_CACHE_TTL = 24 * 3600
_EMBEDDING_DIM = 512


class _InMemoryRedis:
    """Async dict stand-in for Redis (same interface as PHT:03)."""
    def __init__(self) -> None:
        self._store: dict[str, tuple[bytes, float]] = {}

    async def get(self, key: str) -> bytes | None:
        v, exp = self._store.get(key, (None, 0.0))
        return v if (v is not None and exp > time.time()) else None

    async def set(self, key: str, value: bytes, ex: int) -> None:
        self._store[key] = (value, time.time() + ex)

    async def delete(self, key: str) -> None:
        self._store.pop(key, None)


def _clip_encode_text(text: str) -> np.ndarray:
    """Stub CLIP text encoder — returns a deterministic random unit vector so
    the same text always encodes to the same vector (helps the cache test)."""
    rng = np.random.default_rng(abs(hash(text)) % (2**31))
    v = rng.standard_normal(_EMBEDDING_DIM).astype(np.float32)
    return v / np.linalg.norm(v)


def _embed_cache_key(query_text: str) -> str:
    return "embed:" + hashlib.sha256(query_text.encode("utf-8")).hexdigest()[:16]


async def embed_query_cached(
    cache: _InMemoryRedis,
    embed_fn,
    query_text: str,
) -> np.ndarray:
    """Return the CLIP embedding of query_text, served from cache on hit.
    `embed_fn` is the stub CLIP text encoder (or the real one in production)."""
    ck = _embed_cache_key(query_text)
    cached_bytes = await cache.get(ck)
    if cached_bytes is not None:
        return np.frombuffer(cached_bytes, dtype=np.float32).copy()
    vec = embed_fn(query_text).astype(np.float32)
    await cache.set(ck, vec.tobytes(), ex=_EMBED_CACHE_TTL)
    return vec


# Demonstrate: same query twice → second call is served from cache.
cache = _InMemoryRedis()
call_count = [0]
def counting_embed(text: str) -> np.ndarray:
    call_count[0] += 1
    return _clip_encode_text(text)

v1 = await embed_query_cached(cache, counting_embed, "beach with children")
v2 = await embed_query_cached(cache, counting_embed, "beach with children")
print(f"embed_fn call count: {call_count[0]}  (expected 1 — cache hit on second)")
print(f"vectors identical:   {np.allclose(v1, v2)}")
print(f"vector shape:        {v1.dtype} {v1.shape}")


## SQL Candidate Generation

Stage (c): the structured filter becomes a `WHERE` clause. This stage was notebook 13's whole contribution; we keep it as-is. The output is a `set[photo_id]` of candidates — typically hundreds to tens of thousands — that the CLIP ranker will narrow.

:::{.callout-note}
The **(c) → (d) interaction is the biggest latency lever**. If the structured filter is restrictive (e.g. date is one week of 50K photos), candidate count is ~30 and CLIP ranks 30 vectors in microseconds. If there is no structured filter (pure CLIP search), stage (c) returns the entire database and stage (d) is HNSW over the full index — ~5 ms per notebook 12. The hybrid pipeline benefits the first case; the second just falls back to pure CLIP. Both work.

:::


In [ ]:
from sqlalchemy import select, and_


# Shape-only ORM stubs for faces and photos.
class FaceORM:
    person_name = None
    photo_id    = None


class PhotoORM:
    photo_id    = None
    taken_at    = None
    location    = None
    sha256      = None


def build_sql_filter(sq: SearchQuery):
    """Build a SQLAlchemy WHERE clause from a SearchQuery. Returns SQLAlchemy
    expression usable as `select(...).where(...)`.

    In notebook 13 this was the entire stage (c); here we reuse it verbatim.
    Note: `location` is a column populated from EXIF GPS during indexing.
    """
    conditions = []
    if sq.person_name:
        # The person filter is a JOIN on faces; we expose it as a separate EXISTS.
        # In the production code this is a subquery; here we just include the
        # column reference for completeness.
        conditions.append(FaceORM.person_name == sq.person_name)
    if sq.location_hint:
        conditions.append(PhotoORM.location.ilike(f"%{sq.location_hint}%"))
    if sq.start_date and sq.end_date:
        conditions.append(PhotoORM.taken_at.between(sq.start_date, sq.end_date))
    elif sq.start_date:
        conditions.append(PhotoORM.taken_at >= sq.start_date)
    elif sq.end_date:
        conditions.append(PhotoORM.taken_at <= sq.end_date)
    return and_(*conditions) if conditions else None


# Show the filter from the parsed "Ben at Bondi in July 2023" query.
sq = SearchQuery(person_name="Ben", location_hint="Bondi",
                 start_date=date(2023, 7, 1), end_date=date(2023, 8, 1))
filt = build_sql_filter(sq)
print(f"filter constructed: {filt is not None}")
print("SQL fragment shape would be: WHERE person_name = 'Ben'  AND  "
      "location ILIKE '%Bondi%'  AND  taken_at >= '2023-07-01'  AND  taken_at < '2023-08-01'")


## Hybrid Scoring: The Cost of Pure CLIP

The ranked list from stage (d) is sorted by CLIP similarity to `query_vec`. Pure CLIP misranks in two patterns the hybrid score fixes:

1. **Geographic mismatch.** A photo of a generic beach scores 0.85 cosine to "beach at Bondi" because CLIP sees "beach" — but Bondi and Mar del Plata are the same beach to CLIP. The structure says location must be `Bondi` (from stage (c) the candidate set is already filtered by location, but if the user only mentions location without `Bondi` as a strong hint — like "Bondi, July 2023" — pure CLIP ignores the date entirely).
2. **Date mismatch.** Pure CLIP ignores time. Same beach in 2019 and 2023 score identically.

The hybrid score combines CLIP similarity with **structured match strength** — a 0/1 (or fraction) score per candidate computed from how many of the parsed filter fields match the photo's metadata. The weighted sum balances content semantics against metadata exactness:

$$
\text{score}(q, p) = \alpha \cdot \mathrm{sim}_{\text{CLIP}}(q, p)
                    + (1 - \alpha) \cdot \mathrm{match}_{\text{struct}}(q, p)
                    + \beta \cdot \mathrm{face\_boost}(q, p)
$$

with $\alpha = 0.7$, $\beta = 0.3$. The constants are tunable; the architectural point is that *two signals get combined*, not which numbers win.


In [ ]:
from dataclasses import dataclass


@dataclass
class PhotoMeta:
    photo_id:     str
    taken_at:      datetime
    location:      str | None
    person_names:  set[str]            # all persons detected in this photo


@dataclass
class CandidateWithClip:
    photo_id:    str
    meta:        PhotoMeta
    clip_sim:    float                  # cosine sim ∈ [0,1] (already in [0,1] for normalized embeddings)


def structured_match_score(sq: SearchQuery, meta: PhotoMeta) -> float:
    """Fraction of the structured filter fields that the photo's metadata satisfies.

    A photo that matches location AND date AND person returns 1.0; a photo that
    matches only date returns 0.5 (two of three fields). Empty filter returns 1.0
    (a no-op filter is "match everything").
    """
    if not (sq.person_name or sq.location_hint or sq.start_date or sq.end_date):
        return 1.0   # No structured filter → match is vacuously full.

    fields = 0
    matched = 0
    if sq.person_name:
        fields += 1
        if sq.person_name in meta.person_names:
            matched += 1
    if sq.location_hint:
        fields += 1
        if meta.location and sq.location_hint.lower() in meta.location.lower():
            matched += 1
    if sq.start_date or sq.end_date:
        fields += 1
        if sq.start_date and meta.taken_at.date() >= sq.start_date:
            if sq.end_date and meta.taken_at.date() < sq.end_date:
                matched += 1
            elif sq.end_date is None and meta.taken_at.date() >= sq.start_date:
                matched += 1
        elif sq.end_date and meta.taken_at.date() < sq.end_date:
            matched += 1
        else:
            matched += 0
    if fields == 0:
        return 1.0
    return matched / fields


def face_boost(sq: SearchQuery, meta: PhotoMeta) -> float:
    """Bonus scored when the photo contains the requested person's face.

    Returns 0.0 unless `person_name` is in the photo's set of detected persons.
    The bonus is squared-count scaled so a photo of "Ben alone" beats a "group
    of 10 people including Ben". Here we use the simpler 0-or-1 form.
    """
    if not sq.person_name:
        return 0.0
    return 1.0 if sq.person_name in meta.person_names else 0.0


ALPHA = 0.7                       # CLIP weight
BETA  = 0.3                       # face boost weight
# (1 - ALPHA) = 0.3 weight for structured match (incidental, balanced)


def hybrid_score(sq: SearchQuery, cand: CandidateWithClip) -> float:
    """Combine CLIP similarity and structured match score into a single rank score."""
    s_match = structured_match_score(sq, cand.meta)
    f_boost = face_boost(sq, cand.meta)
    clip_n  = max(0.0, min(1.0, cand.clip_sim))   # clamp to [0,1]
    return ALPHA * clip_n + (1 - ALPHA) * s_match + BETA * f_boost


# Demonstrate: two candidate photos of similar CLIP score, one matching the filter.
sq = SearchQuery(person_name="Ben", location_hint="Bondi",
                 start_date=date(2023, 7, 1), end_date=date(2023, 8, 1))

c_match = CandidateWithClip(
    "p-001",
    PhotoMeta("p-001", datetime(2023, 7, 14, 10, 0, 0), "Bondi Beach, AU", {"Ben"}),
    clip_sim=0.78,
)
c_wrong_loc = CandidateWithClip(
    "p-002",
    PhotoMeta("p-002", datetime(2022, 6, 1, 10, 0, 0), "Copacabana, BR", {"Marco"}),
    clip_sim=0.81,         # slightly better CLIP but fails every structured field
)

print(f"pure CLIP winner: {c_wrong_loc.photo_id}  (sim={c_wrong_loc.clip_sim})")
print(f"hybrid winner:     {c_match.photo_id}      "
      f"(hybrid={hybrid_score(sq, c_match):.3f}  vs.  "
      f"hybrid={hybrid_score(sq, c_wrong_loc):.3f})")


## Burst Dedup with MMR

If the user's camera fires 8 fps at a moment, eight near-identical photos appear in the indexed set. CLIP assigns them all similar embeddings and they co-locate in the top-k — leaving eight slots used by one visual concept. [Maximal Marginal Relevance (MMR)](https://en.wikipedia.org/wiki/Maximal_marginal_relevance) iteratively picks from the candidate set, balancing relevance to the query against **dissimilarity to already-picked items**. We oversample DB top-k by 5× (e.g. take 100 candidates to render 20 results) so MMR has room to skip near-duplicates without running short.

The dissimilarity measure is feature-based. We use the photo's CLIP image embedding (the same one stored in pgvector) — `1 - cosine` between a candidate's image embedding and each already-picked image embedding. With a 5× oversample pool, MMR typically discards about half of the pool as duplicates and the user sees 20 diverse results instead of 8 frames of one moment.


In [ ]:
def mmr_select(
    candidates:    list[CandidateWithClip],     # already sorted by hybrid_score desc
    image_embeddings: dict[str, np.ndarray],     # photo_id → unit-norm CLIP image embedding
    k: int,
    lambda_: float = 0.5,                        # 1.0 = pure relevance; 0.0 = pure diversity
) -> list[CandidateWithClip]:
    """Greedy MMR over a pre-sorted candidate list.

    The first pick is the top candidate (max relevance).
    Each subsequent pick maximizes:
        lambda_ * rel(c) - (1 - lambda_) * max_sim(c, selected)
    where rel(c) is candidate's clip_sim and max_sim is the highest cosine
    similarity between c and any already-selected photo.
    """
    if not candidates:
        return []
    selected: list[CandidateWithClip] = [candidates[0]]
    remaining = list(candidates[1:])

    while len(selected) < k and remaining:
        best_score = -float("inf")
        best_idx = 0
        for i, c in enumerate(remaining):
            c_emb = image_embeddings.get(c.photo_id, np.zeros(_EMBEDDING_DIM, dtype=np.float32))
            max_sim = max(
                float(np.dot(c_emb, image_embeddings[s.photo_id]))
                for s in selected
                if s.photo_id in image_embeddings
            ) or 0.0
            mmr = lambda_ * c.clip_sim - (1 - lambda_) * max_sim
            if mmr > best_score:
                best_score = mmr
                best_idx = i
        selected.append(remaining.pop(best_idx))
    return selected


# Demonstrate: simulate an 8-fps burst of 8 near-identical photos at the top of
# the candidate list. Pure take-top-k returns 8 duplicates; MMR returns 8 unique
# frames plus the next 8 from the oversample pool.
rng = np.random.default_rng(42)
burst_embs = [rng.standard_normal(_EMBEDDING_DIM).astype(np.float32) for _ in range(8)]
burst_embs = [e / np.linalg.norm(e) for e in burst_embs]
# Make them all near-identical (cosine ~0.95) by interpolating toward the first.
burst_embs = [
    (0.95 * burst_embs[0] + 0.05 * e) / np.linalg.norm(0.95 * burst_embs[0] + 0.05 * e)
    for e in burst_embs
]
# 18 total: 8 identical-ish burst photos + 10 distinct ones
supplementary = []
for i in range(10):
    v = burst_embs[0] + rng.standard_normal(_EMBEDDING_DIM).astype(np.float32) * 5.0
    supplementary.append(v / np.linalg.norm(v))

embs_map = {}
candidates = []
# Burst candidates first (higher clip_sim to make pure top-k pick them)
for i, e in enumerate(burst_embs):
    pid = f"burst-{i}"
    embs_map[pid] = e
    candidates.append(CandidateWithClip(pid, PhotoMeta(pid, datetime(2023,7,14),"Bondi",{"Ben"}), clip_sim=0.85))
# Then 10 diverse candidates
for i, e in enumerate(supplementary):
    pid = f"unique-{i}"
    embs_map[pid] = e
    candidates.append(CandidateWithClip(pid, PhotoMeta(pid, datetime(2023,7,14),"Bondi",{"Ben"}), clip_sim=0.70))

# Sort by clip_sim desc to mirror stage (d)
candidates.sort(key=lambda c: c.clip_sim, reverse=True)

pure_top_8 = candidates[:8]
mmr_top_8 = mmr_select(candidates, embs_map, k=8, lambda_=0.5)

pure_burst_count = sum(1 for c in pure_top_8 if c.photo_id.startswith("burst-"))
mmr_burst_count = sum(1 for c in mmr_top_8 if c.photo_id.startswith("burst-"))
print(f"pure top-8: {pure_burst_count} burst frames  (8 expected — burst dominated)")
print(f"MMR top-8:  {mmr_burst_count} burst frames  (1-2 expected — diversity enforced)")


## The Full Smart Search Pipeline

Pulling all stages together, the handler below — extending notebook 13's `SmartSearchHandler` — turns a natural-language query into a ranked, deduped `Page[SearchResult]`.


In [ ]:
from datetime import datetime
from pydantic import BaseModel


class SearchRequest(BaseModel):
    query:  str
    top_k:  int = 20


class SearchResultItem(BaseModel):
    photo_id:  str
    score:     float


async def smart_search(
    request: SearchRequest,
    llm_client,
    embed_cache: _InMemoryRedis,
    embed_fn,
    sql_candidates_fn,        # callable(SearchQuery) -> list[PhotoMeta]
    vector_rank_fn,            # callable(query_vec, candidates) -> list[(photo_id, clip_sim)]
    image_embeddings_fn,       # callable(list[photo_id]) -> dict[photo_id, np.ndarray]
) -> Page[SearchResultItem]:
    # (a) Parse
    pq = await parse_query(llm_client, request.query)
    sq = pq.query

    # (b) Embed (cached)
    query_text_for_emb = sq.freeform or request.query
    query_vec = await embed_query_cached(embed_cache, embed_fn, query_text_for_emb)

    # (c) SQL candidates
    candidates_meta = sql_candidates_fn(sq)
    candidate_ids = {m.photo_id for m in candidates_meta}

    # (d) Vector rank over candidates
    ranked = vector_rank_fn(query_vec, candidate_ids)
    meta_by_id = {m.photo_id: m for m in candidates_meta}

    # (e)+(f) Hybrid score
    cand_with_clip = [
        CandidateWithClip(pid, meta_by_id[pid], sim)
        for pid, sim in ranked
    ]
    cand_with_clip.sort(
        key=lambda c: hybrid_score(sq, c),
        reverse=True,
    )

    # (g) MMR over top oversample × 5
    oversample = max(request.top_k * 5, 50)
    oversample_pool = cand_with_clip[:oversample]
    image_embs = image_embeddings_fn([c.photo_id for c in oversample_pool])
    selected = mmr_select(oversample_pool, image_embs, k=request.top_k, lambda_=0.5)

    items = [SearchResultItem(photo_id=c.photo_id,
                               score=float(hybrid_score(sq, c)))
             for c in selected]
    has_more = len(cand_with_clip) > oversample
    return Page[SearchResultItem](items=items, has_more=has_more, next_cursor=None)


# Demonstrate against a mock LLM and stubbed backend functions.
mock_llm = _mock_llm(json.dumps({
    "person_name": "Ben", "location_hint": "Bondi",
    "start_date": "2023-07-01", "end_date": "2023-08-01",
    "freeform": None, "confidence": 0.9,
}))

result = await smart_search(
    request=SearchRequest(query="Ben at Bondi in July 2023", top_k=8),
    llm_client=mock_llm,
    embed_cache=_InMemoryRedis(),
    embed_fn=_clip_encode_text,
    sql_candidates_fn=lambda sq: [
        PhotoMeta("p-001", datetime(2023, 7, 14, 10, 0), "Bondi Beach, AU", {"Ben"}),
        PhotoMeta("p-002", datetime(2023, 7, 14, 11, 0), "Bondi Beach, AU", {"Ben"}),
        PhotoMeta("p-003", datetime(2023, 7, 15, 10, 0), "Bondi Beach, AU", {"Ben", "Lisa"}),
        PhotoMeta("p-004", datetime(2022, 6,  1, 10, 0), "Bondi Beach, AU", {"Ben"}),
    ],
    vector_rank_fn=lambda vec, ids: [(pid, 0.85) for pid in ids],
    image_embeddings_fn=lambda ids: {pid: np.zeros(_EMBEDDING_DIM, dtype=np.float32) for pid in ids},
)
print(f"results: {len(result.items)}")
for it in result.items:
    print(f"  {it.photo_id}  score={it.score:.3f}")


## Cache Invalidation

CLIP embeddings do not change once computed (the CLIP model is fixed). The only reason to evict a query's cached embedding is a model upgrade — at which point we flip a `clip_model_version` config value that becomes part of the cache key prefix, orphaning the old entries silently. The TTL does the cleanup.

```python
def _embed_cache_key(query_text: str) -> str:
    return f"embed:v{CLIP_MODEL_VERSION}:" + hashlib.sha256(query_text.encode("utf-8")).hexdigest()[:16]
```

:::{.callout-note}
A model upgrade is the **only** cause of cache churn in this layer. We could keep both versions in the cache during a migration — pre-warm the new prefix for the top-100 queries, then flip the version — but at 24 h TTL the simpler "flip and let TTL clean up" works for a hobby-scale family library.

:::


## Schema Additions

Search hardening does not require schema changes beyond what notebook 13 established — every column used here (`location`, `taken_at`, `person_name` via `faces`) is already on `photos` or `faces` from notebook 12. The new work lives in the application layer: hybrid scoring is built in Python over rows already fetched; MMR is built in Python over image vectors already retrieved from pgvector. The **only** schema touch is a model-version label in Redis, which is application config.


## Summary

This notebook hardened the search endpoint along three axes. We added a query→embedding Redis cache to make stage (b) near-instant on repeat queries. We replaced notebook 13's "struct-filter-then-ignore-it" pattern with a hybrid score that combines CLIP similarity, structured field match, and a face-presence boost. We added an MMR dedup pass over the top-k so burst-mode frames do not crowd out diversity. The endpoint still returns the same `Page[SearchResultItem]` envelope introduced in PHT:03 — the client is unchanged.

**What changed relative to notebook 12.** `POST /search/` no longer returns raw CLIP-sorted top-k. It returns CLIP+structured+face-boost hybrid-sorted, MMR-deduped results, served from a Redis cache for the embedding stage. The `SearchQuery` shape from notebook 13 is reused; we add a `confidence` field and use the parse output as a scoring input instead of just a filter predicate.

**What this enables for the rest of PHT.** PHT:07 instruments the search latency budget: parse, embed (cached/hit), SQL, vector rank, hybrid, MMR. Each is now a separately metered stage, so the production trace can point at the slow one. PHT:06's delete cascade invalidates the embedding cache indirectly — search itself never serves stale embeddings, but the *result set* for a query that previously matched a deleted photo will change. Search does not need to know about deletes for correctness; PHT:06 takes care of that.

---



---


■
